# 3.4 Eksik Veri

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/04-missing-values.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Handling Missing Data

Eğitimlerdeki veri ile gerçek dünyadaki veri arasındaki fark, gerçek verinin nadiren temiz ve homojen olmasıdır. Özellikle ilginç veri kümelerinde bir miktar eksik veri bulunur. Daha da karmaşıklaştıran şey, farklı kaynakların eksik veriyi farklı biçimlerde göstermesidir.

Bu bölümde eksik veri için genel düşünceleri, Pandas'ın bunu nasıl temsil ettiğini ve Python'da eksik veriyi işlemek için Pandas'ın yerleşik araçlarını ele alacağız. Kitap boyunca eksik veriyi genel olarak null, NaN veya NA değerleri diye anacağız.

## Eksik Veri Sözleşmelerinde Ödünleşimler

Bir tablo veya DataFrame'de eksik verinin varlığını izlemek için çeşitli yaklaşımlar geliştirilmiştir. Genelde iki stratejiden biri etrafında döner: eksik değerleri genel olarak gösteren bir maske kullanmak veya eksik girişi belirten bir gösterge değeri (sentinel) seçmek.

Maskeleme yaklaşımında maske tamamen ayrı bir Boolean dizi olabilir veya veri gösteriminde bir bit ayrılarak değerin yerel null durumu belirtilebilir.

Sentinel yaklaşımında gösterge, eksik tamsayı için –9999 gibi veriye özgü bir kural veya kayan nokta için NaN (Not a Number) gibi IEEE kayan nokta standardının parçası olan özel bir değer olabilir.

Hiçbir yaklaşım ödünsüz değildir. Ayrı maske dizisi ek Boolean depolama ve hesaplama yükü getirir. Sentinel, temsil edilebilir geçerli değer aralığını daraltır ve NaN gibi özel değerler her veri tipinde olmadığı için ek mantık gerekebilir.

Farklı diller ve sistemler farklı sözleşmeler kullanır: R her veri tipinde ayrılmış bit desenleri kullanır; SciDB her hücreye NA durumu için ek bir bayt ekler.

## Pandas'ta Eksik Veri

Pandas'ın eksik değerleri işleme biçimi, kayan nokta dışı tipler için yerleşik NA kavramı olmayan NumPy paketine bağımlılığıyla sınırlıdır.

R'nin her tip için bit deseni ayırması NumPy'nin 14 temel tamsayı tipi (bit genişliği, işaret, endianness) gibi çok daha fazla tipi desteklemesi nedeniyle hantal olurdu. Tüm NumPy tiplerinde özel bit ayırmak büyük ek yük ve muhtemelen NumPy çatallaması gerektirirdi; 8 bit tamsayılarda bir biti maske olarak kullanmak temsil aralığını ciddi daraltır.

Bu kısıtlar nedeniyle Pandas eksik değerleri iki "modda" saklar ve işler:

Her iki durumda da Pandas API işlemleri eksik girişleri öngörülebilir biçimde işler ve yayar. Seçimlerin nedenini anlamak için None, NaN ve NA ödünleşimlerine kısaca bakalım. Her zamanki gibi NumPy ve Pandas'ı içe aktararak başlayalım:


In [ ]:
# import_np_pd.py
import numpy as np
import pandas as pd



### None Gösterge Değeri Olarak

Bazı veri tiplerinde Pandas gösterge olarak None kullanır. None bir Python nesnesidir; None içeren her dizi dtype=object olmalıdır — Python nesneleri dizisi.

Örneğin None'ı NumPy dizisine geçirirseniz:


In [ ]:
# vals1_none.py
vals1 = np.array([1, None, 2, 3])
vals1



dtype=object, NumPy'nin içerik için çıkardığı en iyi ortak temsilin Python nesneleri olduğu anlamına gelir. None kullanmanın dezavantajı, işlemlerin Python düzeyinde, yerel tiplerdeki hızlı işlemlere göre çok daha yavaş yapılmasıdır:


```python
# timeit_int.py
%timeit np.arange(1E6, dtype=int).sum()
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


```python
# timeit_object.py
%timeit np.arange(1E6, dtype=object).sum()
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


Python None ile aritmetik desteklemediğinden sum veya min gibi toplamalar genelde hata verir:


In [ ]:
# vals1_sum.py
vals1.sum()



> **Not**
>

Bu nedenle Pandas sayısal dizilerinde gösterge olarak None kullanmaz.

### NaN: Eksik Sayısal Veri

Diğer gösterge NaN farklıdır; IEEE kayan nokta standardını kullanan tüm sistemler tarafından tanınan özel bir kayan nokta değeridir:


In [ ]:
# vals2_nan.py
vals2 = np.array([1, np.nan, 3, 4])
vals2



NumPy bu dizi için yerel kayan nokta tipi seçti: nesne dizisinin aksine derlenmiş kodda hızlı işlemler desteklenir. NaN bir veri virüsü gibidir — dokunduğu her işlemin sonucu yine NaN olur:


In [ ]:
# one_plus_nan.py
1 + np.nan



In [ ]:
# zero_times_nan.py
0 * np.nan



Toplamlar tanımlıdır (hata vermez) ancak her zaman yararlı değildir:


In [ ]:
# vals2_agg.py
vals2.sum(), vals2.min(), vals2.max()



NumPy eksik değerleri yok sayan nan* toplama sürümleri sağlar:


In [ ]:
# nan_agg.py
np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)



> **Not**
>

NaN'ın ana dezavantajı özellikle kayan nokta değeri olmasıdır; tamsayı, dize vb. için eşdeğer yoktur.

### Pandas'ta NaN ve None

NaN ve None ikisi de yer bulur; Pandas ikisini neredeyse birbirinin yerine kullanılabilir şekilde işler, uygun yerde dönüştürür:


In [ ]:
# series_nan_none.py
pd.Series([1, np.nan, 2, None])



Sentinel değeri olmayan tiplerde NA varken Pandas otomatik tip yükseltmesi yapar. Tamsayı dizisine np.nan atanırsa kayan noktaya yükseltilir:


In [ ]:
# series_int.py
x = pd.Series(range(2), dtype=int)
x



In [ ]:
# series_int_none.py
x[0] = None
x



Tamsayı dizisinin kayan noktaya yükseltilmesine ek olarak Pandas None'ı NaN'a dönüştürür. R gibi alan dillere göre sihirli görünebilir; pratikte nadiren sorun çıkarır.

NA girildiğinde Pandas yükseltme kuralları:

Pandas'ta dize verisi her zaman object dtype ile saklanır.

## Pandas Nullable Dtype'lar

Erken Pandas sürümlerinde yalnızca NaN ve None sentinel değerleri vardı; örtük tip yükseltmesi (ör. gerçek eksik verili tamsayı dizisi yoktu) zordu. Bunun için nullable dtype'lar eklendi — adları büyük harfle yazılır (pd.Int32 vs np.int32). Geriye uyumluluk için yalnızca açıkça istenince kullanılır.

Üç eksik veri göstergesini içeren tamsayı Series örneği:


In [ ]:
# nullable_int32.py
pd.Series([1, np.nan, 2, None, pd.NA], dtype='Int32')



Bu gösterim bölümün geri kalanındaki tüm işlemlerde diğerleriyle birbirinin yerine kullanılabilir.

## Null Değerler Üzerinde İşlem

Pandas None, NaN ve NA'yı eksik/null için esasen birbirinin yerine kullanılabilir sayar. Kolaylık için şu yöntemler vardır:

Bu bölümü bu rutinlerin kısa gösterimiyle bitiriyoruz.

### Null Değerleri Tespit Etme

isnull ve notnull (veya isna / notna) Boolean maske döndürür:


In [ ]:
# data_series.py
data = pd.Series([1, np.nan, 'hello', None])



In [ ]:
# data_isnull.py
data.isnull()



3.2 Veri İndeksleme ve Seçimi'nde anlatıldığı gibi Boolean maskeler doğrudan Series veya DataFrame indeksi olarak kullanılabilir:


In [ ]:
# data_notnull_mask.py
data[data.notnull()]



DataFrame için de benzer Boolean sonuçlar üretilir.

### Null Değerleri Silme

dropna (NA kaldırır) ve fillna (NA doldurur) vardır. Series için sonuç doğrudandır:


In [ ]:
# data_dropna.py
data.dropna()



DataFrame için daha fazla seçenek vardır. Örnek:


In [ ]:
# df_nan.py
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
df



DataFrame'den tek değer değil yalnızca tüm satır veya sütun silinebilir. dropna birçok seçenek sunar.

Varsayılan olarak herhangi null içeren tüm satırlar silinir:


In [ ]:
# df_dropna_rows.py
df.dropna()



axis=1 veya axis='columns' ile null içeren sütunlar silinir:


In [ ]:
# df_dropna_cols.py
df.dropna(axis='columns')



Bu iyi veriyi de atar; yalnızca tüm veya çoğunlukla NA olan satır/sütunları silmek isteyebilirsiniz — how veya thresh ile.

Varsayılan how='any': herhangi bir null varsa satır/sütun gider. how='all' yalnızca hepsi null ise siler:


In [ ]:
# df_col3_nan.py
df[3] = np.nan
df



In [ ]:
# df_dropna_how_all.py
df.dropna(axis='columns', how='all')



thresh satır/sütunun tutulması için gereken minimum null olmayan sayısını belirtir:


In [ ]:
# df_dropna_thresh.py
df.dropna(axis='rows', thresh=3)



İlk ve son satır yalnızca iki null olmayan değer içerdiği için silindi.

### Null Değerleri Doldurma

Bazen NA silmek yerine geçerli bir değerle değiştirmek istersiniz — sıfır, imputasyon veya interpolasyon. isnull maskesiyle yapılabilir; yaygın olduğu için fillna null değerlerin değiştirildiği kopya döndürür.


In [ ]:
# data_fill_series.py
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'), dtype='Int32')
data



Tek değerle doldurma (ör. sıfır):


In [ ]:
# fillna_zero.py
data.fillna(0)



İleri doldurma (önceki değeri yayma):


In [ ]:
# forward fill
data.fillna(method='ffill')



Geri doldurma (sonraki değeri geriye yayma):


In [ ]:
# back fill
data.fillna(method='bfill')



> **Not**
>

DataFrame'de benzer seçenekler ve doldurmanın yapılacağı axis:


In [ ]:
# df_refill.py
df



In [ ]:
# df_fillna_ffill_axis1.py
df.fillna(method='ffill', axis=1)



İleri doldurmada önceki değer yoksa NA kalır.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Eksik değerli küçük bir DataFrame oluşturup dropna ve fillna(0) sonuçlarını karşılaştırın:
          
      import pandas as pd
import numpy as np
df = pd.DataFrame({'a': [1, np.nan, 3], 'b': [4, 5, np.nan]})
print(df)
print("\ndropna:\n", df.dropna())
print("\nfillna(0):\n", df.fillna(0))

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      isnull() maskesiyle yalnızca eksik olmayan satırları seçin:
          
      import pandas as pd
import numpy as np
s = pd.Series([1, np.nan, 3, None])
print(s[s.notnull()])

> **Not**
>

> **Not**
>
